In [50]:
import glob
import os
import re
import pandas as pd

перетворення файле ексель на табличний вигляд

In [51]:
def process_nbu_file(file_path, report_date):
    # 1. Зчитуємо Excel без заголовків
    df = pd.read_excel(file_path, sheet_name='tabl', header=None)

    # 2. Витягуємо рівні шапки (рядки 5, 6, 7 у файлі Excel -> індекси 4, 5, 6 у Python)
    borrower_types = df.iloc[4].ffill()  # Фізичні особи / Юридичні особи
    currencies = df.iloc[5].ffill()      # усього / національна валюта / іноземна валюта
    risk_classes = df.iloc[6]            # 1, 2, 3, 4, 5

    # 3. Вибираємо лише рядки з даними (починаючи з індексу 8 у Python, тобто з рядка 9 Excel)
    data_df = df.iloc[8:].copy()

    # Стовпчик B (індекс 1) - Банк, Стовпчик C (індекс 2) - Показник
    data_df[1] = data_df[1].ffill()  # Заповнюємо назву банку для другого рядка (Кредитний ризик)

    # ПРИВЕДЕННЯ ДО СТРІЧКИ: Додаємо .astype(str) для уникнення AttributeError
    data_df[1] = data_df[1].astype(str).str.replace(r'^\d+\s+', '', regex=True)
    # 4. Формуємо унікальні назви колонок для значень
    column_names = []
    for i in range(len(df.columns)):
        if i == 0:
            column_names.append('N_zp')
        elif i == 1:
            column_names.append('Bank')
        elif i == 2:
            column_names.append('Metric')
        else:
            b_type = str(borrower_types[i]).strip()
            curr = str(currencies[i]).strip()
            r_class = str(risk_classes[i]).strip()
            column_names.append(f"{b_type}__{curr}__Клас_{r_class}")

    data_df.columns = column_names

    # Видаляємо непотрібний стовпчик N_zp та порожні рядки
    data_df = data_df.drop(columns=['N_zp']).dropna(subset=['Bank', 'Metric'])

    # 5. Перетворюємо з широкого формату в довгий (Unpivot / Melt)
    melted_df = data_df.melt(
        id_vars=['Bank', 'Metric'],
        var_name='Combined_Category',
        value_name='Value'
    )

    # Розбиваємо створену колонку назад на окремі виміри
    split_cols = melted_df['Combined_Category'].str.split('__', expand=True)
    melted_df['Borrower_Type'] = split_cols[0]
    melted_df['Currency'] = split_cols[1]
    melted_df['Risk_Class'] = split_cols[2]

    # Прибираємо тимчасову колонку та додаємо дату
    melted_df = melted_df.drop(columns=['Combined_Category'])
    melted_df['Report_Date'] = pd.to_datetime(report_date)

    # Очищаємо числові значення від можливих пробілів чи тексу
    melted_df['Value'] = pd.to_numeric(melted_df['Value'], errors='coerce').fillna(0)

    return melted_df


file_paths = sorted(glob.glob('data/Kred_class_*.xlsx'))
all_dfs = []


for file in file_paths:
    # Витягуємо дату з назви файлу
    date_str = re.search(r'\d{4}-\d{2}-\d{2}', file).group(0)

    # Обробляємо один місяць
    df_clean = process_nbu_file(file, date_str)
    all_dfs.append(df_clean)

# Об'єднуємо все в один чистий датасет!
df_final = pd.concat(all_dfs, ignore_index=True)

# Перевіримо результат
df_final.head(10)

,Bank,Metric,Value,Borrower_Type,Currency,Risk_Class,Report_Date
0,"2 АТ ""Укрексімбанк""",Сума кредитної заборгованості,2.304624e+05,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
1,"2 АТ ""Укрексімбанк""",Кредитний ризик,1.947169e+04,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
2,"6 АТ ""Ощадбанк""",Сума кредитної заборгованості,2.405892e+07,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
3,"6 АТ ""Ощадбанк""",Кредитний ризик,1.517360e+06,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
4,"29 АТ ""БАНК АЛЬЯНС""",Сума кредитної заборгованості,3.716481e+05,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
5,"29 АТ ""БАНК АЛЬЯНС""",Кредитний ризик,4.310781e+04,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
6,"36 АТ ""Райффайзен Банк""",Сума кредитної заборгованості,6.670036e+06,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
7,"36 АТ ""Райффайзен Банк""",Кредитний ризик,2.437754e+05,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
8,"43 АТ ""АЛЬТБАНК""",Сума кредитної заборгованості,2.193276e+04,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01
9,"43 АТ ""АЛЬТБАНК""",Кредитний ризик,3.076504e+02,"Фізичні особи, у тому числі ФОП",усього,Клас_1,2025-02-01


In [54]:
# Зберігаємо у CSV з кодуванням utf-8-sig (щоб кирилиця правильно відкривалася в Excel)
df_final.to_csv('data_2025.csv', index=False, encoding='utf-8-sig')

In [55]:
# Подивитися всі унікальні дати та кількість рядків для кожної
print(df_final['Report_Date'].value_counts().sort_index())

Report_Date
2025-02-01    12810
2025-03-01    12810
2025-04-01    12600
2025-05-01    12600
2025-06-01    12600
2025-07-01    12600
2025-08-01    12600
2025-09-01    12600
2025-10-01    12600
2025-11-01    12600
2025-12-01    12600
2026-01-01    12390
Name: count, dtype: int64


Додає маркер Asset_Status Працюючий / Непрацюючий (NPL)

In [57]:
def classify_npl_exact(row):
    # Приводимо все до нижнього регістру та перетворюємо на текст
    r_class = str(row['Risk_Class']).lower().strip()
    b_type = str(row['Borrower_Type']).lower().strip()

    # Перевірка для Юридичних осіб (шукаємо 10 клас)
    if 'юрид' in b_type or 'корпоратив' in b_type:
        if '10' in r_class:
            return 'Непрацюючий (NPL)'

    # Перевірка для Фізичних осіб та Банків (шукаємо 5 клас або маркер "непрацюючі")
    if 'фізич' in b_type or 'банк' in b_type:
        if '5' in r_class or 'непрацююч' in r_class:
            return 'Непрацюючий (NPL)'

    # Додаткова загальна перевірка на випадок залишків у назвах класів
    if (
        'непрацююч' in r_class
        or r_class.endswith('5')
        or r_class.endswith('10')
        or '5.0' in r_class
        or '10.0' in r_class
    ):
        return 'Непрацюючий (NPL)'

    return 'Працюючий'


In [58]:
df_prep = df_final.copy()
# Додаємо статус активу до кожного рядка
df_prep['Asset_Status'] = df_prep.apply(classify_npl_exact, axis=1)

# Створюємо прапорець для Ощадбанку
df_prep['Is_Oschad'] = df_prep['Bank'].str.contains('Ощадбанк', case=False, na=False)

In [59]:
import numpy as np
# 3. Агрегуємо та розраховуємо % NPL
df_debts = df_prep[
    df_prep['Metric'].str.contains('заборгован', case=False, na=False)
]

summary_df = (
    df_debts.groupby(['Report_Date', 'Is_Oschad', 'Asset_Status'])['Value']
    .sum()
    .unstack(fill_value=0)
)

for col in ['Працюючий', 'Непрацюючий (NPL)']:
    if col not in summary_df.columns:
        summary_df[col] = 0

summary_df['Усього_Портфель'] = (
    summary_df['Працюючий'] + summary_df['Непрацюючий (NPL)']
)

summary_df['%_NPL'] = np.where(
    summary_df['Усього_Портфель'] > 0,
    (summary_df['Непрацюючий (NPL)'] / summary_df['Усього_Портфель']) * 100,
    0,
)

print('Результат розрахунку:')
print(summary_df[['Працюючий', 'Непрацюючий (NPL)', 'Усього_Портфель', '%_NPL']])

Результат розрахунку:
Asset_Status              Працюючий  Непрацюючий (NPL)  Усього_Портфель  \
Report_Date Is_Oschad                                                     
2025-02-01  False      1.519058e+09       1.409794e+09     2.928851e+09   
            True       1.574171e+08       3.181243e+08     4.755414e+08   
2025-03-01  False      1.544843e+09       1.414124e+09     2.958967e+09   
            True       1.586129e+08       3.172638e+08     4.758767e+08   
2025-04-01  False      1.590429e+09       1.372279e+09     2.962708e+09   
            True       1.653469e+08       3.149982e+08     4.803450e+08   
2025-05-01  False      1.633264e+09       1.377649e+09     3.010913e+09   
            True       1.637323e+08       3.170783e+08     4.808105e+08   
2025-06-01  False      1.661971e+09       1.387634e+09     3.049605e+09   
            True       1.668981e+08       3.152157e+08     4.821137e+08   
2025-07-01  False      1.743836e+09       1.398323e+09     3.142159e+09   
   

In [60]:
# Зберігаємо готовий датасет поруч із файлом додатка
df_prep.to_csv('data_ready_for_charts.csv', index=False, encoding='utf-8-sig')